# Setup

Here the required imports are listed, as well as paths to the data and where to store the schemas.

In [2]:
# All necessary imports listed here:
from pathlib import Path
import re
from typing import Dict, Optional, List, Any, Tuple, Sequence, Union
import hashlib
from datetime import datetime
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import numpy as np
from dataclasses import dataclass
import faiss
from sentence_transformers import SentenceTransformer
import pickle
import math
import torch

DATA_DIR = Path("./data/")
SCHEMA_DIR = Path("./schema/")

# Load and Preprocess

Firstly the functions to process the data are listed, and secondly the code to parse and chunk all documents.

In [3]:
# Functions for preprocessing:
def stable_id(text: str, prefix: str = "doc") -> str:
    h = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:16]
    return f"{prefix}_{h}"

def normalize_whitespace(raw_str: str) -> str:
    """
    Normalize the whitespaces over a string.
    """
    raw_str = raw_str.replace("\r\n", "\n").replace("\r", "\n")
    raw_str = re.sub(r"[ \t]+", " ", raw_str)
    raw_str = re.sub(r"\n{3,}", "\n\n", raw_str)
    return raw_str.strip()

def split_sections(raw: str, sections: List[str]) -> Dict[str, str]:
    """
    Split name, ingredients, info, instructions sections.
    """
    raw = normalize_whitespace(raw)
    
    pattern = re.compile(           # section header pattern
        r"^(name|ingredients|info|instructions)\s*:\s*$", re.IGNORECASE | re.MULTILINE
        )

    matches = list(pattern.finditer(raw))
    if not matches:
        # If no headers found, treat entire thing as instructions (fallback)
        return {"name": "", "ingredients": "", "info": "", "instructions": raw}

    sections: Dict[str, str] = {key: "" for key in sections}

    for idx, match in enumerate(matches):
        key = match.group(1).lower()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(raw)
        content = raw[start:end].strip()
        sections[key] = content

    return sections

def parse_minutes(value: str) -> Optional[int]:
    """
    Convert time indications to minutes, returns 'None' if string is not parsable
    """
    value = value.strip().lower()

    # Common patterns
    # "60 min" / "60 mins"
    mins = re.search(r"(\d+)\s*(min|mins|minute|minutes)\b", value)
    # "1 h 15 min"
    hrs = re.search(r"(\d+)\s*(h|hr|hrs|hour|hours)\b", value)

    if not mins and not hrs:
        return None

    minutes = 0
    if hrs:
        minutes += int(hrs.group(1)) * 60
    if mins:
        minutes += int(mins.group(1))

    return minutes if minutes > 0 else None

def parse_servings(value: str) -> Optional[int]:
    """
    Parse the no. of servings stated in the recipe.
    """
    v = value.strip()
    m = re.search(r"(\d+)", v)
    return int(m.group(1)) if m else None

def parse_info_block(info_text: str) -> Dict[str, Any]:
    """
    Parses lines like:
      Difficulty: Average
      Prep time: 60 min
      Cook time: 45 min
      Serving: 8
      Cost: Average
      Note + chilling time...
    """
    out: Dict[str, Any] = {
        "difficulty": None,
        "cost": None,
        "prep_time": None,
        "cook_time": None,
        "total_time": None,
        "servings": None,
        "notes": []
    }

    lines = [ln.strip() for ln in info_text.split("\n") if ln.strip()]
    for ln in lines:
        # Key: Value
        kv = re.match(r"^([A-Za-z \-\+]+)\s*:\s*(.+)$", ln)
        if kv:
            key = kv.group(1).strip().lower()
            val = kv.group(2).strip()

            if key in {"difficulty"}:
                out["difficulty"] = val
            elif key in {"cost"}:
                out["cost"] = val
            elif key in {"prep time", "prep_time"}:
                out["prep_time"] = parse_minutes(val)
            elif key in {"cook time", "cook_time"}:
                out["cook_time"] = parse_minutes(val)
            elif key in {"serving", "servings", "yield"}:
                out["servings"] = parse_servings(val)
            else:
                # Preserve unknown structured lines as notes
                out["notes"].append(ln)
        else:
            # Free-form note line
            out["notes"].append(ln)

    # Compute total_time if possible
    if out["prep_time"] is not None or out["cook_time"] is not None:
        out["total_time"] = (out["prep_time"] or 0) + (out["cook_time"] or 0)

    return out

def parse_ingredients_block(ingredients_text: str) -> List[Dict[str, str]]:
    """
    Parses ingredient lines like:
    Type 00 flour: 3cups(380 g)
    Nutmeg: to taste
    """
    NAME_PART  = r"^\s*(.+?)\s*"
    SEP_PART   = r"\s*:\s*"
    VALUE_PART = r"(.+?)\s*$"
    INGREDIENT_LINE_RE = re.compile(f"{NAME_PART}{SEP_PART}{VALUE_PART}")
    items: List[Dict[str, str]] = []
    for line in ingredients_text.split("\n"):

        line = line.strip()
        if not line:                # whitespace
            continue
        match = INGREDIENT_LINE_RE.match(line)
        if match:
            name = match.group(1).strip()
            amount = match.group(2).strip()
        else:                       # no match -> consider line as name
            name, amount = line, ""
        items.append({"name": name, "amount": amount})
    return items

def split_instructions(instructions_text: str) -> List[str]:
    """
    Split line into steps. Split newline when present, or split by sentence boundary.
    """
    text = normalize_whitespace(instructions_text)

    # Separate newlines if present.
    if "\n" in instructions_text:
        parts = [normalize_whitespace(p) for p in instructions_text.split("\n") if p.strip()]
        # Merge very short fragments into previous step
        steps: List[str] = []
        for p in parts:
            if steps and len(p) < 40:
                steps[-1] = (steps[-1] + " " + p).strip()
            else:
                steps.append(p)
        return steps

    # Or do a sentence split if newlines are not present.
    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]

    # Group into 2-3 sentences per step
    steps: List[str] = []
    buf: List[str] = []
    for s in sents:
        buf.append(s)
        if len(buf) >= 3:
            steps.append(" ".join(buf))
            buf = []
    if buf:
        steps.append(" ".join(buf))
    return steps

def infer_tags(dish_name: str, ingredients: List[Dict[str, str]], instructions: str) -> List[str]:
    name_l = (dish_name or "").lower()
    ing_l = " ".join([i["name"].lower() for i in ingredients])
    txt = (name_l + " " + ing_l + " " + instructions.lower())

    tags = set()

    if any(k in txt for k in ["tart", "cake", "cookie", "dessert", "chocolate", "sugar", "icing", "ganache"]):
        tags.add("dessert")
    if any(k in txt for k in ["pasta", "spaghetti", "penne", "rigatoni", "tagliatelle"]):
        tags.add("pasta")
    if any(k in txt for k in ["fish", "salmon", "tuna", "anchovy", "shrimp", "prawn", "octopus", "mussel"]):
        tags.add("seafood")
    if any(k in txt for k in ["beef", "pork", "chicken", "lamb", "sausage", "guanciale", "bacon"]):
        tags.add("meat")

    # Very bold veggie tag adding
    if "meat" not in tags and "seafood" not in tags:
        tags.add("vegetarian_candidate")

    return sorted(tags)

def build_document_schema(raw: str, sections: List[str], url: Optional[str] = None) -> Dict[str, Any]:
    sections = split_sections(raw, sections)

    dish_name = sections.get("name", "").strip()
    ingredients = parse_ingredients_block(sections.get("ingredients", ""))
    info = parse_info_block(sections.get("info", ""))
    steps = split_instructions(sections.get("instructions", ""))

    doc_id = stable_id(dish_name + "\n" + raw, prefix="doc")

    # Normalize into schema
    doc: Dict[str, Any] = {
        "doc_id": doc_id,
        "url": url,
        "dish_name": dish_name or None,
        "region": None,     # Future extension
        "doc_type": "recipe",
        "sections": {
            "ingredients": [f'{x["name"]}: {x["amount"]}'.strip(": ").strip() for x in ingredients],
            "steps": steps,
            "notes": "\n".join(info.get("notes", [])).strip() or None,
            "history": None,
            "variations": None,
            "techniques": None,
        },
        "servings": info.get("servings"),
        "prep_time": info.get("prep_time"),
        "cook_time": info.get("cook_time"),
        "total_time": info.get("total_time"),
        "tags": infer_tags(dish_name, ingredients, sections.get("instructions", "")),
        "parsed_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }

    return doc

def make_chunks(doc: Dict[str, Any], max_chars: int = 1200) -> List[Dict[str, Any]]:
    """
    Create section-aware chunks suitable for hybrid retrieval.
    Uses a simple char budget
    """
    chunks: List[Dict[str, Any]] = []

    def add_chunk(section: str, text: str, idx: int, extra: Optional[Dict[str, Any]] = None):
        c = {
            "chunk_id": f'{doc["doc_id"]}#{section}#{idx}',
            "doc_id": doc["doc_id"],
            "dish_name": doc.get("dish_name"),
            "region": doc.get("region"),
            "doc_type": doc.get("doc_type"),
            "source": doc.get("source"),
            "url": doc.get("url"),
            "section": section,
            "tags": doc.get("tags", []),
            "text": text.strip(),
        }
        if extra:
            c.update(extra)
        chunks.append(c)

    # Ingredients: usually one chunk
    ing = doc["sections"].get("ingredients") or []
    if ing:
        ing_text = f"the ingredients for {doc.get('dish_name')} are:" + "\n".join(f"- {x}" for x in ing)
        add_chunk("ingredients", ing_text, 0)

    # Notes: one chunk
    notes = doc["sections"].get("notes")
    if notes:
        add_chunk("notes", "Notes:\n" + notes, 0)

    # Steps: chunk in groups
    steps = doc["sections"].get("steps") or []
    if steps:
        buf: List[str] = []
        idx = 0
        for i, step in enumerate(steps, start=1):
            candidate = "\n".join(buf + [f"{i}. {step}"]).strip()
            if buf and len(candidate) > max_chars:
                add_chunk("steps", "Instructions:\n" + "\n".join(buf), idx, extra={"step_start": i - len(buf), "step_end": i - 1})
                idx += 1
                buf = [f"{i}. {step}"]
            else:
                buf.append(f"{i}. {step}")

        if buf:
            start = int(re.match(r"^(\d+)\.", buf[0]).group(1)) if re.match(r"^(\d+)\.", buf[0]) else None
            end = int(re.match(r"^(\d+)\.", buf[-1]).group(1)) if re.match(r"^(\d+)\.", buf[-1]) else None
            add_chunk("steps", "Instructions:\n" + "\n".join(buf), idx, extra={"step_start": start, "step_end": end})

    # Times/servings as a compact factual chunk
    facts = []
    if doc.get("servings") is not None:
        facts.append(f"Servings: {doc['servings']}")
    if doc.get("prep_time") is not None:
        facts.append(f"Prep time (min): {doc['prep_time']}")
    if doc.get("cook_time") is not None:
        facts.append(f"Cook time (min): {doc['cook_time']}")
    if doc.get("total_time") is not None:
        facts.append(f"Total time (min): {doc['total_time']}")
    if facts:
        add_chunk("info", f"Info for {doc.get('dish_name')}:\n" + "\n".join(f"- {x}" for x in facts), 0)

    return chunks

def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for rec in records:
            file.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [4]:
MAX_CHUNK_CHARS = 60
SECTION_ORDER = ["name", "ingredients", "info", "instructions"]

# List files in data directory.
filepaths: List[Path] = []
docs = Path(DATA_DIR)
filepaths.extend(sorted([filepath for filepath in docs.glob("**/*.txt") if filepath.is_file()]))

documents: List[Dict[str, Any]] = []
chunks: List[Dict[str, Any]] = []

# Read each data file.
for file in filepaths:
    raw = read_text_file(file)
    doc = build_document_schema(raw=raw, sections=SECTION_ORDER, url=None)
    # keep original file
    doc["source_file"] = str(file)
    documents.append(doc)
    chunks.extend(make_chunks(doc, max_chars=MAX_CHUNK_CHARS))

out_dir = Path(SCHEMA_DIR)
write_jsonl(out_dir / "documents.jsonl", documents)
write_jsonl(out_dir / "chunks.jsonl", chunks)

print(f"Wrote {len(documents)} documents to {out_dir / 'documents.jsonl'}")
print(f"Wrote {len(chunks)} chunks to {out_dir / 'chunks.jsonl'}")

Wrote 6642 documents to schema/documents.jsonl
Wrote 87147 chunks to schema/chunks.jsonl


# Embedding
Using BM25 and dense vectors using FAISS.

In [5]:
@dataclass(frozen=True)
class Chunk:
    text: str
    metadata: Dict[str, Any]

def _coerce_metadata(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    Accepts either:
      (A) {"text": "...", "metadata": {...}}
      (B) flat: {"text": "...", "chunk_id": "...", "doc_id": "...", ...}
    Returns metadata dict.
    """
    if "metadata" in obj and isinstance(obj["metadata"], dict):
        md = dict(obj["metadata"])
        # keep any other top-level fields (except text/metadata) as well
        for k, v in obj.items():
            if k not in {"text", "metadata"} and k not in md:
                md[k] = v
        return md

    # flat case: everything except text is metadata
    md = {k: v for k, v in obj.items() if k != "text"}
    return md

def load_chunks(data_path: str, limit: Optional[int] = None) -> List[Chunk]:
    """
    Loads JSONL where each line is a chunk-like object.
    Works with:
      - flat chunk lines (your example)
      - lines with {"text": "...", "metadata": {...}}
    """
    path = Path(data_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix.lower() != ".jsonl":
        raise ValueError("This loader expects a .jsonl file.")

    out: List[Chunk] = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and len(out) >= int(limit):
                break

            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            if not isinstance(obj, dict):
                continue

            text = (obj.get("text") or "").strip()
            if not text:
                continue

            md = _coerce_metadata(obj)
            md.setdefault("row_index", i)

            # Handy defaults for your schema
            md.setdefault("doc_id", obj.get("doc_id"))
            md.setdefault("chunk_id", obj.get("chunk_id"))
            md.setdefault("dish_name", obj.get("dish_name"))
            md.setdefault("section", obj.get("section"))
            md.setdefault("doc_type", obj.get("doc_type"))
            md.setdefault("source", obj.get("source"))

            out.append(Chunk(text=text, metadata=md))

    if not out:
        raise ValueError("No valid chunks found (no non-empty 'text' fields).")

    return out

In [6]:

@dataclass
class SimilarityIndex:
    index: faiss.Index
    embeddings: np.ndarray | None
    model_name: str

def _encode_texts(
    model: SentenceTransformer,
    texts: Sequence[str],
    batch_size: int = 64,
    device: str = "cpu",
) -> np.ndarray:
    emb = model.encode(
        list(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,  # cosine similarity via inner product
        show_progress_bar=True,
        device=device,
    )
    emb = emb.astype("float32", copy=False)
    return emb

def build_similarity_index(
    texts: Sequence[str],
    enc_model_name: str = "all-MiniLM-L6-v2",
    device: str = "cpu",
    batch_size: int = 64,
) -> Tuple[SentenceTransformer, SimilarityIndex]:
    """
    Builds a FAISS index for cosine similarity using normalized embeddings and IndexFlatIP.
    """
    model = SentenceTransformer(enc_model_name, device=device)
    embeddings = _encode_texts(model, texts, batch_size=batch_size, device=device)

    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)

    return model, SimilarityIndex(index=index, embeddings=embeddings, model_name=enc_model_name)

def search(
    model: SentenceTransformer,
    sim: SimilarityIndex,
    query: str,
    top_k: int = 5,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns (scores, indices) each of shape (top_k,).
    Scores are cosine similarities in [-1, 1] (usually [0,1] with these models).
    """
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idx = sim.index.search(q, top_k)
    return scores[0], idx[0]

In [7]:

_WORD_RE = re.compile(r"[A-Za-z0-9]+")

def tokenize(text: str) -> List[str]:
    return [m.group(0).lower() for m in _WORD_RE.finditer(text or "")]

@dataclass
class BM25Index:
    k1: float
    b: float
    avgdl: float
    doc_len: List[int]
    df: Dict[str, int]                 # document frequency
    idf: Dict[str, float]              # idf per term
    tfs: List[Dict[str, int]]          # term freq per document

def build_bm25(
    texts: Sequence[str],
    k1: float = 1.5,
    b: float = 0.75,
) -> BM25Index:
    tfs: List[Dict[str, int]] = []
    df: Dict[str, int] = {}
    doc_len: List[int] = []

    for text in texts:
        toks = tokenize(text)
        doc_len.append(len(toks))
        tf: Dict[str, int] = {}
        for t in toks:
            tf[t] = tf.get(t, 0) + 1
        tfs.append(tf)

        # update df with unique terms
        for t in tf.keys():
            df[t] = df.get(t, 0) + 1

    N = len(texts)
    avgdl = (sum(doc_len) / N) if N else 0.0

    # BM25+ style IDF (common stable variant)
    idf: Dict[str, float] = {}
    for term, n_qi in df.items():
        # log( (N - df + 0.5) / (df + 0.5) + 1 )
        idf[term] = math.log((N - n_qi + 0.5) / (n_qi + 0.5) + 1.0)

    return BM25Index(k1=k1, b=b, avgdl=avgdl, doc_len=doc_len, df=df, idf=idf, tfs=tfs)

def bm25_search(
    index: BM25Index,
    query: str,
    top_k: int = 50,
) -> Tuple[List[float], List[int]]:
    q_terms = tokenize(query)
    if not q_terms:
        return [], []

    scores: List[float] = [0.0] * len(index.tfs)

    for term in q_terms:
        idf = index.idf.get(term)
        if idf is None:
            continue

        for doc_id, tf in enumerate(index.tfs):
            f = tf.get(term, 0)
            if f == 0:
                continue

            dl = index.doc_len[doc_id]
            denom = f + index.k1 * (1.0 - index.b + index.b * (dl / index.avgdl if index.avgdl else 0.0))
            score = idf * (f * (index.k1 + 1.0)) / (denom if denom else 1.0)
            scores[doc_id] += score

    # top-k by score
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    ranked = [(i, s) for i, s in ranked if s > 0.0][:top_k]

    idx = [i for i, _ in ranked]
    sc = [s for _, s in ranked]
    return sc, idx

In [8]:
INDEX_PATH = SCHEMA_DIR / "chunks.faiss"
META_PATH  = SCHEMA_DIR / "chunks_meta.pkl"
CHUNKS_PATH = SCHEMA_DIR / "chunks.jsonl"

device = "cuda:0" if torch.cuda.is_available() else "cpu"

def save_store(
    *,
    index: faiss.Index,
    chunks: List[Chunk],
    enc_model_name: str,
    bm25: BM25Index,
) -> None:
    SCHEMA_DIR.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(INDEX_PATH))

    payload = {
        "enc_model_name": enc_model_name,
        "chunks": chunks,
        "bm25": bm25,
    }
    with META_PATH.open("wb") as f:
        pickle.dump(payload, f)

def load_store(device: str = "cpu") -> Tuple[SentenceTransformer, faiss.Index, List[Chunk], str, BM25Index]:
    if not INDEX_PATH.exists() or not META_PATH.exists():
        raise FileNotFoundError(
            f"Missing saved store. Expected {INDEX_PATH} and {META_PATH}."
        )

    index = faiss.read_index(str(INDEX_PATH))

    with META_PATH.open("rb") as f:
        payload: Dict[str, Any] = pickle.load(f)

    enc_model_name = payload["enc_model_name"]
    chunks = payload["chunks"]
    bm25 = payload["bm25"]

    embed_model = SentenceTransformer(enc_model_name, device=device)
    return embed_model, index, chunks, enc_model_name, bm25

In [9]:
p = Path(CHUNKS_PATH)
print("exists:", p.exists(), "is_file:", p.is_file(), "size:", p.stat().st_size if p.exists() else None)
chunks = load_chunks(CHUNKS_PATH)
texts = [c.text for c in chunks]

# Dense index
embed_model, dense_sim = build_similarity_index(
    texts=texts,
    enc_model_name="all-MiniLM-L6-v2",
    device=device,
    batch_size=16,  # safer on macOS
)

# BM25 index
bm25 = build_bm25(texts)

# Save both
save_store(
    index=dense_sim.index,
    chunks=chunks,
    enc_model_name=dense_sim.model_name,
    bm25=bm25,
)

dense = dense_sim

exists: True is_file: True size: 46974008


Batches:   0%|          | 0/5447 [00:00<?, ?it/s]

# Run

## Hybrid Retrieval

In [10]:

@dataclass
class HybridConfig:
    k_bm25: int = 100
    k_dense: int = 100
    rrf_k: int = 60          # RRF constant, typical 60
    top_k: int = 10          # final results

def rrf_fuse(
    bm25_ranks: Sequence[int],
    dense_ranks: Sequence[int],
    rrf_k: int = 60,
) -> List[int]:
    """
    Reciprocal Rank Fusion:
      score(d) = Σ 1 / (rrf_k + rank_method(d))
    rank is 1-based.
    """
    scores: Dict[int, float] = {}

    for r, doc_id in enumerate(bm25_ranks, start=1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + r)

    for r, doc_id in enumerate(dense_ranks, start=1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + r)

    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

def hybrid_search(
    *,
    embed_model,
    dense: SimilarityIndex,
    bm25: BM25Index,
    query: str,
    cfg: HybridConfig,
) -> List[int]:
    # BM25 candidates (ranked)
    _, bm25_idx = bm25_search(bm25, query=query, top_k=cfg.k_bm25)

    # Dense candidates (ranked)
    _, dense_idx = search(embed_model, dense, query=query, top_k=cfg.k_dense)
    dense_idx_list = [int(i) for i in dense_idx.tolist() if int(i) >= 0]

    fused = rrf_fuse(bm25_idx, dense_idx_list, rrf_k=cfg.rrf_k)
    return fused[: cfg.top_k]

## Query Rewriting

In [11]:
class DialogueState:
    dish: Optional[str] = None
    intent: Optional[str] = None
    constraints: Dict[str, str] = []
    history: List[str] = []


class DialogueStateTracking:
    def __init__(self, device: str = "cpu", model_name: str = "google/flan-t5-large"):
        self.state = DialogueState()

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

        self.pipe = pipeline(
            "text2text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if device == "cuda" else -1,
        )

    def _generate(self, prompt: str, max_new_tokens: int = 64) -> str:
        return self.pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
        )[0]["generated_text"].strip()

    def update_state(self, user_input: str) -> None:
        self.state.history.append(user_input)

        prompt = (
            f"""
            Extract the dish mentioned in the conversation
            Conversation:\n{self.state.history[-1]}
            Return ONLY the dish name or 'none'.
            """
        )
        dish = self._generate(prompt)
        if dish.lower() != "none":
            self.state.dish = dish
        return self.state.dish

    def rewrite_for_retrieval(self, user_query: str) -> str:
        # Make the query standalone for RAG retrieval
        dish = self.state.dish or "the discussed dish"
        prompt = (
            f"""
            Rewrite the user query into a single, standalone search query.
            Dish: {dish}
            User query: {user_query}
            Return ONLY the rewritten search query.
            """
        )
        return self._generate(prompt)


dst = DialogueStateTracking(device="cpu")
def rewrite_query(query: str) -> str:
    query = dst.rewrite_for_retrieval(query)
    dst.update_state(query)
    return query

Device set to use cpu


## Metadata Filter
This part assigns tags to the query using a cosine similarity between the tag and a description of the tags. It also applies a negation if present in the query. Excluding tags that follow a negation, to avoid "no meat" scoring high for a meat tag because of semantic similarity in the words.

In [12]:
# metadata filter

TagScores = List[Tuple[str, float]]

_NEGATION_PATTERNS: Dict[str, List[re.Pattern]] = {
    # If query indicates "no meat", penalize the "meat" tag
    "meat": [
        re.compile(r"\b(no|without|w\/o|not)\s+(any\s+)?meat\b", re.I),
        re.compile(r"\b(meat[-\s]?free|no\s+meat)\b", re.I),
        re.compile(r"\b(no|without|w\/o|not)\s+(any\s+)?(bacon|ham|beef|pork|chicken)\b", re.I),
        re.compile(r"\b(vegetarian|vegan)\b", re.I),  # optional: treat vegetarian/vegan as excluding meat
    ],
    # If query indicates "no seafood", penalize the "seafood" tag
    "seafood": [
        re.compile(r"\b(no|without|w\/o|not)\s+(any\s+)?(seafood|fish|shellfish)\b", re.I),
        re.compile(r"\b(seafood[-\s]?free|fish[-\s]?free)\b", re.I),
    ],
    # Optional examples:
    # "pasta": [re.compile(r"\b(gluten[-\s]?free|no\s+gluten)\b", re.I)],
}

def apply_negation_gate(
    query: str,
    scores: TagScores,
    penalty: float = 0.15,   # multiply forbidden tag score by this (0.0 blocks, 0.1..0.3 soft)
) -> TagScores:
    score_map = dict(scores)

    for tag, patterns in _NEGATION_PATTERNS.items():
        if tag not in score_map:
            continue
        if any(p.search(query) for p in patterns):
            score_map[tag] *= penalty

    gated = list(score_map.items())
    gated.sort(key=lambda x: x[1], reverse=True)
    return gated

def get_all_tag_categories_from_jsonl(jsonl_path: str) -> List[str]:
    tags: Set[str] = set()
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}") from e
            obj_tags = obj.get("tags")
            if obj_tags:
                tags.update(obj_tags)
    return sorted(tags)

def encode_texts(
    model: SentenceTransformer,
    texts: Sequence[str],
    batch_size: int = 64,
    device: str = "cpu",
    template: Optional[str] = None,
) -> np.ndarray:
    if template is not None:
        texts = [template.format(text=t) for t in texts]

    emb = model.encode(
        list(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,  # dot == cosine
        show_progress_bar=False,
        device=device,
    )
    return emb.astype(np.float32, copy=False)

def filter_top_idxs_by_tags(
    top_idxs: list[int],
    chunks,
    picked_tags: list[str],
    keep_k: int = 10,
) -> list[int]:
    """
    Keep only chunks whose metadata has at least one of picked_tags.
    If nothing matches, return the original top_idxs (fallback).
    """
    if not picked_tags:
        return top_idxs[:keep_k]

    picked = set(picked_tags)

    kept: list[int] = []
    for i in top_idxs:
        md = chunks[i].metadata or {}
        tags = md.get("tags", [])
        if any(t in picked for t in tags):
            kept.append(i)
            if len(kept) >= keep_k:
                break

    return kept if kept else top_idxs[:keep_k]


def l2_normalize(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v if n == 0 else (v / n)


WeightedSyn = Union[str, Tuple[str, float]]  # ("synonym", weight)


def build_tag_vectors(
    model: SentenceTransformer,
    tag_to_synonyms: Dict[str, Sequence[WeightedSyn]],
    batch_size: int = 64,
    device: str = "cpu",
    template: str = "This recipe is {text}.",
) -> Dict[str, np.ndarray]:
    """
    Build one stable vector per tag by:
      1) encoding all synonyms (batched)
      2) weighted mean per tag
      3) L2 normalization
    """

    # Flatten all synonyms so we encode once
    flat_texts: List[str] = []
    flat_meta: List[Tuple[str, float]] = []  # (tag, weight)

    for tag, syns in tag_to_synonyms.items():
        cleaned: List[Tuple[str, float]] = []
        for s in syns:
            if isinstance(s, tuple):
                text, w = s
            else:
                text, w = s, 1.0
            text = text.strip()
            if text:
                cleaned.append((text, float(w)))

        if not cleaned:
            raise ValueError(f"Tag '{tag}' has no synonyms.")

        for text, w in cleaned:
            flat_texts.append(text)
            flat_meta.append((tag, w))

    # Encode once
    all_emb = encode_texts(model, flat_texts, batch_size=batch_size, device=device, template=template)

    # Aggregate per tag
    tag_sum: Dict[str, np.ndarray] = {}
    tag_wsum: Dict[str, float] = {}

    for (tag, w), vec in zip(flat_meta, all_emb):
        if tag not in tag_sum:
            tag_sum[tag] = np.zeros_like(vec)
            tag_wsum[tag] = 0.0
        tag_sum[tag] += w * vec
        tag_wsum[tag] += w

    tag_vectors: Dict[str, np.ndarray] = {}
    for tag in tag_sum:
        v = tag_sum[tag] / max(tag_wsum[tag], 1e-12)
        tag_vectors[tag] = l2_normalize(v.astype(np.float32, copy=False))

    return tag_vectors


def score_query_against_tags(
    model: SentenceTransformer,
    query: str,
    tag_vectors: Dict[str, np.ndarray],
    device: str = "cpu",
    template: str = "User request: {text}",
) -> List[Tuple[str, float]]:
    q = encode_texts(model, [query], batch_size=1, device=device, template=template)[0]
    scores = [(tag, float(q @ vec)) for tag, vec in tag_vectors.items()]
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


def top_k(scores: List[Tuple[str, float]], k: int = 5) -> List[Tuple[str, float]]:
    return scores[:k]

def pick_top_k_with_margin(scores: TagScores, k: int = 2, margin: float = 0.03) -> TagScores:
    if not scores:
        return []
    picked = [scores[0]]
    for tag, s in scores[1:]:
        if len(picked) >= k:
            break
        if picked[0][1] - s <= margin:
            picked.append((tag, s))
    return picked


device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Print tags found in your dataset
print(get_all_tag_categories_from_jsonl("./schema/chunks.jsonl"))

TAG_SYNONYMS: Dict[str, List[WeightedSyn]] = {
    "vegetarian_candidate": [
        ("vegetarian", 2.0),
        ("meatless", 1.5),
        ("no meat", 1.5),
        ("plant-based", 1.2),
        ("veggie", 1.0),
        ("vegetable dish", 1.0),
        # ("without meat", 1.5),
    ],
    "dessert": [
        ("dessert", 2.0),
        ("sweet dish", 1.5),
        ("cake", 1.0),
        ("pudding", 1.0),
        ("treat", 0.8),
        ("after dinner sweet", 1.0),
    ],
    "meat": [
        ("with meat", 2.0),
        ("with beef", 1.0),
        ("with pork", 1.0),
        ("with chicken", 1.0),
        ("with ham", 1.0),
        ("with bacon", 1.0),
        ("minced meat", 0.8),
        ("contains meat", 1.5)
    ],
    "pasta": [
        ("pasta", 2.0),
        ("pasta dish", 1.4),
        ("spaghetti", 1.0),
        ("penne", 1.0),
        ("lasagna", 1.0),
        ("macaroni", 0.8),
        ("fettuccine", 0.8),
    ],
    "seafood": [
        ("seafood", 2.0),
        ("fish", 1.5),
        ("shellfish", 1.0),
        ("shrimp", 1.0),
        ("salmon", 1.0),
        ("tuna", 0.9),
        ("mussels", 0.8),
    ],
    "ingredients": [
        ("ingredients", 2.0),
        ("contains", 2.0),
        ("ingredient", 2.0),
        ("goes in", 1.0)
    ]
}

tag_vecs = build_tag_vectors(
    model,
    TAG_SYNONYMS,
    device=device,
    template="This recipe is {text}.",
)

# showcase

queries = [
    "I want something without meat",
    "a sweet dish after dinner",
    "a sweet dish",
    "A carbonara without meat",
    "recipe with bacon and ham",
    "What are the ingredients to tiramisu?",
    "What are the ingredients to lasagna?",
]

for q in queries:
    scores = score_query_against_tags(
        model, q, tag_vecs, device=device, template="User request: {text}"
    )

    scores = apply_negation_gate(q, scores, penalty=0.10)  # strong penalty
    # print(top_k(scores, k=5))
    print(f"\nQuery: {q}")
    for tag, s in top_k(scores, k=5):
        print(f"  {tag:22s}  {s:.3f}")

    picked = pick_top_k_with_margin(scores, k=2, margin=0.03)
    print("  picked:", picked)   

def get_tags(query):
    scores = score_query_against_tags(
        model, query, tag_vecs, device=device, template="User request: {text}"
    )

    scores = apply_negation_gate(query, scores, penalty=0.10)  # strong penalty
    # print(top_k(scores, k=5))
    print(f"\nQuery: {query}")
    for tag, s in top_k(scores, k=5):
        print(f"  {tag:22s}  {s:.3f}")

    picked = pick_top_k_with_margin(scores, k=2, margin=0.03)
    return [tag for tag, score in picked]

def overlap(li1, li2):
    for el in li1:
        if el in li2:
            return True
    return False

def filter_tags(tags, chunks, candidate_idx):
    out_chunks = []
    for chunk_idx in candidate_idx:
        chunk_tags = chunks[chunk_idx].metadata.get('tags')
        section = chunks[chunk_idx].metadata.get('section')
        if overlap(tags, chunk_tags) or section in tags:
            out_chunks.append(chunks[chunk_idx])
    return out_chunks

['dessert', 'meat', 'pasta', 'seafood', 'vegetarian_candidate']

Query: I want something without meat
  vegetarian_candidate    0.548
  seafood                 0.388
  ingredients             0.326
  pasta                   0.297
  dessert                 0.295
  picked: [('vegetarian_candidate', 0.5477765798568726)]

Query: a sweet dish after dinner
  dessert                 0.708
  pasta                   0.534
  seafood                 0.509
  meat                    0.506
  ingredients             0.490
  picked: [('dessert', 0.7076407074928284)]

Query: a sweet dish
  dessert                 0.630
  seafood                 0.482
  pasta                   0.448
  ingredients             0.440
  vegetarian_candidate    0.436
  picked: [('dessert', 0.6302295923233032)]

Query: A carbonara without meat
  vegetarian_candidate    0.525
  seafood                 0.373
  pasta                   0.335
  ingredients             0.315
  dessert                 0.282
  picked: [('vegetarian_c

## Generative Model

In [13]:
class GenerativeQA:
    def __init__(self, device: str = "cpu", model_name: str = "google/flan-t5-xl") -> None:
        device_obj = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device_obj)

        self.pipe = pipeline(
            "text2text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if device_obj.type == "cuda" else -1,
        )

    def answer(self, question: str, contexts: list[str]) -> str:
        """
        Answer a question given a list of retrieved context chunks.
        """
        joined_context = "\n".join(contexts)

        prompt = (
            f"question: {question}\n"
            f"context: {joined_context}\n\n"
            "Answer the question using ONLY the information provided in the context."
            "If the answer cannot be derived from the context, say that the context does not contain the required information. "
        )
        out = self.pipe(
            prompt,
            max_new_tokens=250,
            num_beams=4,
            do_sample=False,
        )[0]["generated_text"]
        return out.strip()
    
    def call_llm(self, prompt: str, *, max_new_tokens: int = 80, num_beams: int = 1) -> str:
        out = self.pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3
        )[0]["generated_text"]
        return out.strip()

## Loop to Run Conversation

In [14]:

cfg = HybridConfig(
    k_bm25=100,  # candidates from BM25
    k_dense=100, # candidates from dense
    rrf_k=60,
    top_k=30,    # final top-k returned
)
# generate response
gen = GenerativeQA(device=device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [19]:
while True:
    query = input("\nQuery> ").strip()
    if not query:
        continue
    if query.lower() in {"quit", "exit"}:
        break

    #rewrite query
    query = rewrite_query(query)
    print(query)

    top_idxs = hybrid_search(
        embed_model=embed_model,
        dense=dense,
        bm25=bm25,
        query=query,
        cfg=cfg,
    )

    # filter results
    tags = get_tags(query)
    print(f"tags: {tags}")
    filtered_chunks = filter_tags(tags, chunks, top_idxs)

    print("\nTop results:")
    for c in filtered_chunks:
        md = c.metadata
        text_preview = c.text.replace("\n", " ")[:140]
        print(
            # f"{rank:2d}. "
            f"doc_id={md.get('doc_id')} | "
            f"chunk_id={md.get('chunk_id')} | "
            f"section={md.get('section')} | "
            f"tags = {md.get('tags')} | "
            f"dish_name={md.get('dish_name')}"
        )
        print(f"    {text_preview}...\n")

    context = [c.text for c in filtered_chunks[:3]]
    answer = gen.answer(question=query, contexts=context)
    print(answer)
    print(f"Retrieved from: {[fc.metadata.get('doc_id') for fc in filtered_chunks[:3]]}")


Query> What are the ingredients to lasagna?
What are the ingredients to lasagna?

Query: What are the ingredients to lasagna?
  ingredients             0.525
  pasta                   0.523
  meat                    0.449
  dessert                 0.412
  vegetarian_candidate    0.408
tags: ['ingredients', 'pasta']

Top results:
doc_id=doc_49a0ab303e4dbfcf | chunk_id=doc_49a0ab303e4dbfcf#ingredients#0 | section=ingredients | tags = ['meat', 'pasta'] | dish_name=Lasagna soup
    the ingredients for Lasagna soup are:- Lasagne pasta: 6oz(170 g)- (lasagna noodles, mafalde, reginelle) - Sausage: 1lb(450 g)- Italian (casi...

doc_id=doc_e10f4c4b7e49d3d5 | chunk_id=doc_e10f4c4b7e49d3d5#ingredients#0 | section=ingredients | tags = ['dessert', 'pasta', 'vegetarian_candidate'] | dish_name=Lasagna with Swiss Chard
    the ingredients for Lasagna with Swiss Chard are:- Green Lasagna: 0.5lb(200 g) - Chard: 7oz(200 g)- (to clean) - Mozzarella cheese: 1 ¾cup(2...

doc_id=doc_53a54f68084f1440 | chunk

Lasagne pasta: 6oz(170 g)- (lasagna noodles, mafalde, reginelle) - Sausage: 1lb(450 g)- Italian (casing removed) - Extra virgin olive oil: 2tbsp - Onions: 1- chopped - Garlic: 1clove- minced - Dried oregano: 1tsp - Tomato paste: 2tbsp - Beef broth: 4cups(900 ml) - Canned tomatoes: 1(450 g) - Ricotta cheese: 12cup(120 g) - Parmigiano Reggiano PDO cheese: 14cup(30 g) - Parsley: a few- fresh - Salt: to taste
Retrieved from: ['doc_49a0ab303e4dbfcf', 'doc_e10f4c4b7e49d3d5', 'doc_53a54f68084f1440']

Query> What is the white sauce in it?
What is the white sauce in lasagna?

Query: What is the white sauce in lasagna?
  pasta                   0.571
  ingredients             0.497
  meat                    0.495
  dessert                 0.479
  seafood                 0.470
tags: ['pasta']

Top results:
doc_id=doc_4f45de47fbca6c6d | chunk_id=doc_4f45de47fbca6c6d#steps#0 | section=steps | tags = ['dessert', 'meat', 'pasta'] | dish_name=White Lasagna
    Instructions: 1. White lasagna is a tasty

béchamel
Retrieved from: ['doc_4f45de47fbca6c6d', 'doc_4f45de47fbca6c6d', 'doc_64ce0c661088c6af']

Query> Is there a gluten free varaint of it?
Is there a gluten free version of lasagna?

Query: Is there a gluten free version of lasagna?
  pasta                   0.446
  vegetarian_candidate    0.376
  meat                    0.375
  ingredients             0.359
  dessert                 0.310
tags: ['pasta']

Top results:
doc_id=doc_918e23a5a9694d83 | chunk_id=doc_918e23a5a9694d83#steps#3 | section=steps | tags = ['dessert', 'pasta', 'vegetarian_candidate'] | dish_name=Colorful lasagna
    Instructions: 4. It's also a smart pick if you’re searching for a healthy lasagna recipe that’s meat-free and packed with veggies. Some town...

doc_id=doc_4f45de47fbca6c6d | chunk_id=doc_4f45de47fbca6c6d#ingredients#0 | section=ingredients | tags = ['dessert', 'meat', 'pasta'] | dish_name=White Lasagna
    the ingredients for White Lasagna are:- Lasagne egg pasta: 8.8oz(250 g) - Sausage: 0.9lb(400

Lasagne pasta: 6oz(170 g)- (lasagna noodles, mafalde, reginelle) - Sausage: 1lb(450 g)- Italian (casing removed) - Extra virgin olive oil: 2tbsp - Onions: 1- chopped - Garlic: 1clove- minced - Dried oregano: 1tsp - Tomato paste: 2tbsp - Beef broth: 4cups(900 ml) - Canned tomatoes: 1(450 g) - Ricotta cheese: 12cup(120 g) - Parmigiano Reggiano PDO cheese: 14cup(30 g) - Parsley: a few- fresh - Salt: to taste
Retrieved from: ['doc_49a0ab303e4dbfcf', 'doc_e10f4c4b7e49d3d5', 'doc_53a54f68084f1440']

Query> How do I make tiramisu?
How do I make tiramisu?

Query: How do I make tiramisu?
  dessert                 0.240
  pasta                   0.212
  seafood                 0.207
  vegetarian_candidate    0.194
  ingredients             0.183
tags: ['dessert', 'pasta']

Top results:
doc_id=doc_fcff71c9e3db6021 | chunk_id=doc_fcff71c9e3db6021#steps#8 | section=steps | tags = ['dessert', 'vegetarian_candidate'] | dish_name=Cardamom Tiramisu
    Instructions: 9. You can also make the tiramisu wi

You can also make the tiramisu with an eggless Mascarpone cream Instructions: 2.
Retrieved from: ['doc_fcff71c9e3db6021', 'doc_626f054f676c5522', 'doc_76de74c4f7978295']

Query> How long does it take in the fridge?
How long does tiramisu take in the fridge?

Query: How long does tiramisu take in the fridge?
  dessert                 0.206
  seafood                 0.185
  ingredients             0.177
  vegetarian_candidate    0.175
  pasta                   0.171
tags: ['dessert', 'seafood']

Top results:
doc_id=doc_7ecd76225ef62b34 | chunk_id=doc_7ecd76225ef62b34#steps#28 | section=steps | tags = ['dessert', 'vegetarian_candidate'] | dish_name=Gourmet tiramisu
    Instructions: 29. You can keep the tiramisu for up to 3 days in the fridge....

doc_id=doc_88b95f7123d29759 | chunk_id=doc_88b95f7123d29759#steps#16 | section=steps | tags = ['dessert', 'pasta', 'vegetarian_candidate'] | dish_name=Tiramisù cream phyllo cups
    Instructions: 17. You can store tiramisu cream in the fridge, c

3 days
Retrieved from: ['doc_7ecd76225ef62b34', 'doc_88b95f7123d29759', 'doc_f21d84aa3d810b69']


KeyboardInterrupt: Interrupted by user

# Evaluation 
Evaluation of model using evaluation_model2.py evaluation set

## Metrics

These include f1 accuracy, f1, cosine similarity, and for retrieval: mrr, precision, recall, hit, ndcg

In [16]:

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

def token_f1(pred: str, gold: str) -> float:
    pt = _norm(pred).split()
    gt = _norm(gold).split()
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    from collections import Counter
    pc, gc = Counter(pt), Counter(gt)
    common = sum((pc & gc).values())
    if common == 0: return 0.0
    prec = common / len(pt)
    rec  = common / len(gt)
    return 2*prec*rec/(prec+rec)

def list_f1(pred_items, gold_items) -> float:
    p = {_norm(x) for x in pred_items if _norm(x)}
    g = {_norm(x) for x in gold_items if _norm(x)}
    if not p and not g: return 1.0
    if not p or not g: return 0.0
    inter = len(p & g)
    prec = inter / len(p)
    rec  = inter / len(g)
    return 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)

def answer_accuracy(pred: str, expected):
    # Heuristic accuracy in [0,1]
    pred_n = _norm(pred)
    if isinstance(expected, list):
        # parse a list-ish answer from model output
        cand = [x.strip() for x in re.split(r"[\n,;•\-\*]+", pred) if x.strip()]
        f1 = list_f1(cand, expected)
        return 1.0 if f1 >= 0.70 else f1

    exp_n = _norm(str(expected))
    if exp_n in {"yes","no"}:
        if pred_n.startswith("yes") or pred_n in {"y","true"}: pred_yn="yes"
        elif pred_n.startswith("no") or pred_n in {"n","false"}: pred_yn="no"
        else: pred_yn = pred_n.split(" ")[0] if pred_n else ""
        return 1.0 if pred_yn==exp_n else 0.0

    if exp_n and exp_n in pred_n:
        return 1.0

    f1 = token_f1(pred, str(expected))
    return 1.0 if f1 >= 0.80 else f1

def mrr_at_k(ranked, gold, k):
    gold=set(gold)
    for i,x in enumerate(ranked[:k], start=1):
        if x in gold: return 1.0/i
    return 0.0

def precision_at_k(ranked, gold, k):
    if k<=0: return 0.0
    gold=set(gold)
    return sum(1 for x in ranked[:k] if x in gold)/k

def recall_at_k(ranked, gold, k):
    gold=set(gold)
    if not gold: return 0.0
    return sum(1 for x in ranked[:k] if x in gold)/len(gold)

def hit_at_k(ranked, gold, k):
    gold=set(gold)
    return 1.0 if any(x in gold for x in ranked[:k]) else 0.0

def ndcg_at_k(ranked, gold, k):
    gold=set(gold)
    dcg=0.0
    for i,x in enumerate(ranked[:k], start=1):
        if x in gold:
            dcg += 1.0/math.log2(i+1)
    ideal = min(len(gold), k)
    idcg = sum(1.0/math.log2(i+1) for i in range(1, ideal+1))
    return (dcg/idcg) if idcg>0 else 0.0

def cosine_sim(encoder, a: str, b: str) -> float:
    ea = encoder.encode([a], normalize_embeddings=True)
    eb = encoder.encode([b], normalize_embeddings=True)
    return float(np.dot(ea[0], eb[0]))

def _load_json_list(p: Path):
    data = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(data, list): return [str(x) for x in data]
    if isinstance(data, dict):
        for key in ("ids","doc_ids","chunk_ids","gold_doc_id","gold_chunk_ids"):
            if key in data and isinstance(data[key], list):
                return [str(x) for x in data[key]]
    raise ValueError(f"Expected list json at {p}")

def resolve_gold_ids(item: dict, base_dir: Path, field: str, file_field: str = None):
    """Find the gold ids for questions (in gold sets)"""
    val = item.get(field, None)
    file_path = item.get(file_field, None) if file_field else None

    if isinstance(val, list):
        if len(val)==0 and isinstance(file_path,str) and file_path:
            return _load_json_list((base_dir/file_path).resolve())
        return [str(x) for x in val]

    if isinstance(val, str) and val:
        if val.endswith(".json"):
            return _load_json_list((base_dir/val).resolve())
        return [val]

    if (val is None or val=="") and isinstance(file_path,str) and file_path:
        return _load_json_list((base_dir/file_path).resolve())

    return []


# Set path

In [17]:
import os

PROJECT_ROOT = Path.cwd().parent.resolve()
os.chdir(PROJECT_ROOT)

print("Working directory set to:", Path.cwd())

# sanity checks (relative to PROJECT_ROOT)
assert Path("notebook/schema").exists(), "notebook/schema not found"
assert Path("evaluation").exists(), "evaluation/ folder not found"

EVAL_PATH = Path("evaluation/evaluation_model2.json")   
OUT_JSON = Path("evaluation/eval_results_3.json")
K = 10 
FALLBACK_IF_EMPTY_CONTEXT = True

raw = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
eval_data = raw["questions"]
base_dir = EVAL_PATH.parent


Working directory set to: /scratch/s5228786/language-technology-practical


## Run evaluation

- There are no generation metrics calculated if a question only has an expected behavior. These questions only have a manually scored accuracy. 
- In the evaluation, pre and post-filter metrics are calculated for diagnostic purposes
- Overall evaluation focuses on end-to-end performance
- Generation is only done with filtered contexts

In [20]:
gen = GenerativeQA(device=device)

results = []
doc_rows_pre, chunk_rows_pre = [], []
doc_rows_post, chunk_rows_post = [], []
gen_rows = []


def aggregate(rows, key):
    return float(np.mean([r.get(key,0.0) for r in rows])) if rows else 0.0

for item in eval_data:
    item_id = item.get("id")
    item_type = str(item.get("type","")).lower()

    def eval_question(q: str, expected_answer=None, gold_doc_item=None, gold_chunk_item=None):
        q_orig = q
        q_used = rewrite_query(q_orig)

        top_idxs = hybrid_search(
            embed_model=embed_model,
            dense=dense,
            bm25=bm25,
            query=q_used,
            cfg=HybridConfig(k_bm25=100, k_dense=100, rrf_k=60, top_k=K),
        )

        ranked_doc_ids_pre = [chunks[i].metadata.get("doc_id") for i in top_idxs]
        ranked_chunk_ids_pre = [chunks[i].metadata.get("chunk_id") for i in top_idxs]

        # Deduplicate doc ranking (keep first occurrence / best rank)
        ranked_doc_ids_pre_dedup = []
        seen = set()
        for d in ranked_doc_ids_pre:
            if d not in seen:
                ranked_doc_ids_pre_dedup.append(d)
                seen.add(d)

        # Tag filter 
        tags = get_tags(q_used)
        filtered_chunks = filter_tags(tags, chunks, top_idxs)

        # Fallback
        if FALLBACK_IF_EMPTY_CONTEXT and not filtered_chunks:
            filtered_chunks = [chunks[i] for i in top_idxs]

        ranked_doc_ids_post = [c.metadata.get("doc_id") for c in filtered_chunks]
        ranked_chunk_ids_post = [c.metadata.get("chunk_id") for c in filtered_chunks]

        ranked_doc_ids_post_dedup = []
        seen = set()
        for d in ranked_doc_ids_post:
            if d not in seen:
                ranked_doc_ids_post_dedup.append(d)
                seen.add(d)
        
        # metrics (doc/chunk) pre + post
        doc_metrics_pre = {}
        chunk_metrics_pre = {}
        doc_metrics_post = {}
        chunk_metrics_post = {}

        if gold_doc_item:
            doc_metrics_pre = {
                "mrr": mrr_at_k(ranked_doc_ids_pre_dedup, gold_doc_item, K),
                "precision": precision_at_k(ranked_doc_ids_pre_dedup, gold_doc_item, K),
                "recall": recall_at_k(ranked_doc_ids_pre_dedup, gold_doc_item, K),
                "hit": hit_at_k(ranked_doc_ids_pre_dedup, gold_doc_item, K),
                "ndcg": ndcg_at_k(ranked_doc_ids_pre_dedup, gold_doc_item, K),
            }
            doc_metrics_post = {
                "mrr": mrr_at_k(ranked_doc_ids_post_dedup, gold_doc_item, K),
                "precision": precision_at_k(ranked_doc_ids_post_dedup, gold_doc_item, K),
                "recall": recall_at_k(ranked_doc_ids_post_dedup, gold_doc_item, K),
                "hit": hit_at_k(ranked_doc_ids_post_dedup, gold_doc_item, K),
                "ndcg": ndcg_at_k(ranked_doc_ids_post_dedup, gold_doc_item, K),
            }

        if gold_chunk_item:
            chunk_metrics_pre = {
                "mrr": mrr_at_k(ranked_chunk_ids_pre, gold_chunk_item, K),
                "precision": precision_at_k(ranked_chunk_ids_pre, gold_chunk_item, K),
                "recall": recall_at_k(ranked_chunk_ids_pre, gold_chunk_item, K),
                "hit": hit_at_k(ranked_chunk_ids_pre, gold_chunk_item, K),
                "ndcg": ndcg_at_k(ranked_chunk_ids_pre, gold_chunk_item, K),
            }
            chunk_metrics_post = {
                "mrr": mrr_at_k(ranked_chunk_ids_post, gold_chunk_item, K),
                "precision": precision_at_k(ranked_chunk_ids_post, gold_chunk_item, K),
                "recall": recall_at_k(ranked_chunk_ids_post, gold_chunk_item, K),
                "hit": hit_at_k(ranked_chunk_ids_post, gold_chunk_item, K),
                "ndcg": ndcg_at_k(ranked_chunk_ids_post, gold_chunk_item, K),
            }

        # Generation
        context_texts = [c.text for c in filtered_chunks]
        pred = gen.answer(question=q_used, contexts=context_texts)

        gen_metrics = {}
        if expected_answer is not None:
            acc = answer_accuracy(pred, expected_answer)
            f1 = acc if isinstance(expected_answer, list) else token_f1(pred, str(expected_answer))
            gold_text = "; ".join(map(str, expected_answer)) if isinstance(expected_answer, list) else str(expected_answer)
            cos = cosine_sim(embed_model, pred, gold_text)
            gen_metrics = {"accuracy": acc, "f1": f1, "cosine_similarity": cos}

        # Add rows for aggregation
        if doc_metrics_pre: doc_rows_pre.append(doc_metrics_pre)
        if chunk_metrics_pre: chunk_rows_pre.append(chunk_metrics_pre)
        if doc_metrics_post: doc_rows_post.append(doc_metrics_post)
        if chunk_metrics_post: chunk_rows_post.append(chunk_metrics_post)
        if gen_metrics: gen_rows.append(gen_metrics)

        top_contexts = [{
            "rank": r+1,
            "chunk_id": c.metadata.get("chunk_id"),
            "doc_id": c.metadata.get("doc_id"),
            "section": c.metadata.get("section"),
            "tags": c.metadata.get("tags"),
            "dish_name": c.metadata.get("dish_name"),
            "text": c.text
        } for r,c in enumerate(filtered_chunks)]

        return {
            "question_original": q_orig,
            "question_used": q_used,
            "tags": tags,
            "expected_answer": expected_answer,
            "prediction": pred,
            "ranked_doc_ids_pre_filter": ranked_doc_ids_pre,
            "ranked_chunk_ids_pre_filter": ranked_chunk_ids_pre,
            "ranked_doc_ids_post_filter": ranked_doc_ids_post,
            "ranked_chunk_ids_post_filter": ranked_chunk_ids_post,
            "doc_metrics_pre_filter": doc_metrics_pre,
            "chunk_metrics_pre_filter": chunk_metrics_pre,
            "doc_metrics_post_filter": doc_metrics_post,
            "chunk_metrics_post_filter": chunk_metrics_post,
            "gen_metrics": gen_metrics,
            "top_contexts": top_contexts,
        }

    # Handle multi-turn questions
    if item_type == "multi-turn" or "turns" in item:
        parent_gold_docs = resolve_gold_ids(item, base_dir, field="gold_doc_id", file_field="gold_doc_id_file")
        turn_out = []
        for t_i, turn in enumerate(item.get("turns", [])):
            q = turn.get("text")
            expected = turn.get("expected_answer", None)
            gold_chunks = resolve_gold_ids(turn, base_dir, field="gold_chunk_ids")
            turn_res = eval_question(q, expected_answer=expected, gold_doc_item=parent_gold_docs, gold_chunk_item=gold_chunks)
            turn_res["turn_index"] = t_i
            turn_out.append(turn_res)
        results.append({"id": item_id, "type": item.get("type"), "turns": turn_out})
    else:
        q = item.get("question")
        expected = item.get("expected_answer", None)
        gold_docs = resolve_gold_ids(item, base_dir, field="gold_doc_id", file_field="gold_doc_id_file")
        gold_chunks = resolve_gold_ids(item, base_dir, field="gold_chunk_ids")
        res = eval_question(q, expected_answer=expected, gold_doc_item=gold_docs, gold_chunk_item=gold_chunks)
        res["id"] = item_id
        res["type"] = item.get("type")
        results.append(res)

summary = {
    "k": K,
    "fallback_if_empty_context": FALLBACK_IF_EMPTY_CONTEXT,
    "doc_retrieval_pre_filter": {
        "MRR@k": aggregate(doc_rows_pre, "mrr"),
        "Precision@k": aggregate(doc_rows_pre, "precision"),
        "Recall@k": aggregate(doc_rows_pre, "recall"),
        "Hit@k": aggregate(doc_rows_pre, "hit"),
        "nDCG@k": aggregate(doc_rows_pre, "ndcg"),
        "n_items": len(doc_rows_pre),
    },
    "chunk_retrieval_pre_filter": {
        "MRR@k": aggregate(chunk_rows_pre, "mrr"),
        "Precision@k": aggregate(chunk_rows_pre, "precision"),
        "Recall@k": aggregate(chunk_rows_pre, "recall"),
        "Hit@k": aggregate(chunk_rows_pre, "hit"),
        "nDCG@k": aggregate(chunk_rows_pre, "ndcg"),
        "n_items": len(chunk_rows_pre),
    },
    "doc_retrieval_post_filter": {
        "MRR@k": aggregate(doc_rows_post, "mrr"),
        "Precision@k": aggregate(doc_rows_post, "precision"),
        "Recall@k": aggregate(doc_rows_post, "recall"),
        "Hit@k": aggregate(doc_rows_post, "hit"),
        "nDCG@k": aggregate(doc_rows_post, "ndcg"),
        "n_items": len(doc_rows_post),
    },
    "chunk_retrieval_post_filter": {
        "MRR@k": aggregate(chunk_rows_post, "mrr"),
        "Precision@k": aggregate(chunk_rows_post, "precision"),
        "Recall@k": aggregate(chunk_rows_post, "recall"),
        "Hit@k": aggregate(chunk_rows_post, "hit"),
        "nDCG@k": aggregate(chunk_rows_post, "ndcg"),
        "n_items": len(chunk_rows_post),
    },
    "generation": {
        "Accuracy": aggregate(gen_rows, "accuracy"),
        "F1": aggregate(gen_rows, "f1"),
        "CosineSimilarity": aggregate(gen_rows, "cosine_similarity"),
        "n_items": len(gen_rows),
    },
}

out = {"summary": summary, "results": results}
OUT_JSON.write_text(json.dumps(out, indent=2), encoding="utf-8")

print("Wrote:", OUT_JSON)
print(json.dumps(summary, indent=2))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
Token indices sequence length is longer than the specified maximum sequence length for this model (850 > 512). Running this sequence through the model will result in indexing errors



Query: What ingredients are in Roasted Pumpkin Cream with Crunchy Vegetables?
  ingredients             0.547
  dessert                 0.489
  vegetarian_candidate    0.481
  meat                    0.446
  pasta                   0.361

Query: What ingredients are in Cream Cake?
  ingredients             0.579
  dessert                 0.565
  pasta                   0.395
  meat                    0.381
  vegetarian_candidate    0.356

Query: Which ingredients do I need to make gluten-free tiramisu?
  ingredients             0.437
  vegetarian_candidate    0.382
  meat                    0.359
  dessert                 0.316
  pasta                   0.308

Query: Does the Christmas Crepes contain eggs?
  ingredients             0.370
  seafood                 0.313
  dessert                 0.305
  vegetarian_candidate    0.297
  meat                    0.295

Query: Is saffron used in Orange Saffron Risotto?
  ingredients             0.323
  pasta                   0.283
  vegeta


Query: Should I store them in the fridge?
  dessert                 0.119
  seafood                 0.118
  ingredients             0.102
  vegetarian_candidate    0.069
  pasta                   0.066

Query: How many days can I store them for?
  dessert                 0.014
  seafood                 -0.023
  ingredients             -0.026
  pasta                   -0.040
  vegetarian_candidate    -0.078

Query: How many eggs do I need for Chestnut Flour Cantucci with Almonds and Chocolate?
  ingredients             0.344
  dessert                 0.320
  vegetarian_candidate    0.274
  meat                    0.263
  pasta                   0.254

Query: What kind of tomatoes do I need for Roasted Tomato Pesto?
  vegetarian_candidate    0.354
  ingredients             0.304
  meat                    0.303
  pasta                   0.295
  dessert                 0.286

Query: What is the total time I need to make Ciaudedda?
  dessert                 0.150
  ingredients             


Query: What jam should I use for a jam tart?
  meat                    0.186
  ingredients             0.175
  seafood                 0.153
  dessert                 0.139
  pasta                   0.131

Query: Do I need basil or mint for Beetroot Culurgiones with Potatoes and Blue Cheese Sauce?
  ingredients             0.441
  dessert                 0.427
  vegetarian_candidate    0.427
  pasta                   0.378
  meat                    0.361

Query: Should I leave the skin on the pumpkin for pumpkin in the pan?
  dessert                 0.262
  vegetarian_candidate    0.214
  ingredients             0.182
  meat                    0.151
  pasta                   0.114

Query: How many ingredients do I need for red lentil tofu?
  ingredients             0.477
  meat                    0.422
  vegetarian_candidate    0.413
  dessert                 0.359
  seafood                 0.314

Query: What herbs do I need for crispy baked chickpeas?
  vegetarian_candidate    0.462



Query: Do you know a cocktail recipe that uses whiskey?
  ingredients             0.427
  dessert                 0.377
  pasta                   0.375
  meat                    0.368
  seafood                 0.324

Query: Does eggnog use alcohol?
  ingredients             0.407
  seafood                 0.347
  pasta                   0.346
  dessert                 0.312
  vegetarian_candidate    0.311

Query: I don't like brandy, can I use a different liqueur for eggnog
  ingredients             0.334
  dessert                 0.323
  pasta                   0.269
  seafood                 0.235
  meat                    0.209

Query: Are the eggs raw in eggnog?
  ingredients             0.413
  seafood                 0.409
  vegetarian_candidate    0.379
  meat                    0.354
  dessert                 0.330

Query: What is the total time needed to make eggnog?
  ingredients             0.240
  seafood                 0.227
  dessert                 0.225
  pasta       

# Evaluation notes

- This evaluation was run with a fixed configuration and without ablation studies due to runtime constraints (~20 minutes per run).
- Generation evaluation uses a mix of automatic (token F1, cosine similarity) and manual accuracy scores
- For some questions (e.g. procedural instructions, when there is no expected answer), only manual evaluation is done
- When no chunks pass filtering, the system falls back to unfiltered retrieved chunks to avoid empty-context generation. This is a design trade-off that may affect results. 
- Suggestion-style questions (with very large gold sets) are inherently recall-bounded; ranking metrics (e.g. MRR) are more informative than recall for these cases.
